In [ ]:
import pymc as pm
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
from scipy.signal import lfilter

np.random.seed(42)
T = 100
hist_rain = np.random.uniform(0, 20, T)
hist_flow = np.zeros(T)
hist_flow[0] = 10

#  Loop kept for step-wise data generation clarity. # wil switch to vector eventualy

for t in range(1, T):
    hist_flow[t] = 10 + 0.8 * (hist_flow[t-1] - 10) + 0.5 * hist_rain[t] + np.random.normal(0, 1)

# Mean center it (as strictly required for AR)
flow_c = hist_flow - hist_flow.mean()
rain_c = hist_rain - hist_rain.mean()

with pm.Model() as ar_model:
    # MutableData registration for out-of-sample forecasting and modularity
    flow_data = pm.MutableData("flow_data", flow_c[:-1])
    rain_data = pm.MutableData("rain_data", rain_c[1:])
    obs_data = pm.MutableData("obs_data", flow_c[1:])

    # Priors
    rho = pm.Normal("rho", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=1)

    # AR(1) with exogenous variable (Rain)
    mu = rho * flow_data + beta * rain_data

    # Likelihood
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=obs_data)

    # Inference
    trace = pm.sample(1000, tune=1000, return_inferencedata=True, random_seed=42)

    pm.sample_posterior_predictive(trace, extend_inferencedata=True, random_seed=42)

# (Trace and HDI validation)
az.plot_trace(trace)
az.plot_ppc(trace, num_pp_samples=100)
plt.tight_layout()

# (95% Credible Intervals)
summary_df = az.summary(trace, hdi_prob=0.95)
print(summary_df[['mean', 'hdi_2.5%', 'hdi_97.5%']])

# Overlay 95% HDI against Observed Reality
plt.figure(figsize=(10, 4))
y_pred = trace.posterior_predictive["obs"]
az.plot_hdi(np.arange(1, T), y_pred, hdi_prob=0.95, color="gray", smooth=False, fill_kwargs={"alpha": 0.5, "label": "95% HDI"})
plt.plot(np.arange(1, T), flow_c[1:], color="blue", label="Observed Flow (Centered)", lw=1.5)
plt.title("Contractual Risk: Forecasted vs. True Physical Conditions")
plt.legend()
plt.show()

# Clause 4.12: Out-of-Sample Extreme Foreseeability
# Simulating a consecutive 10-day Monsoon Surge (15 units)
with ar_model:
    pm.set_data({
        "flow_data": np.full(10, flow_c[-1]),  # Static lagged flow for simplicity
        "rain_data": np.full(10, 15.0 - hist_rain.mean()),
        "obs_data": np.zeros(10) # Dummy placeholder
    })
    forecast = pm.sample_posterior_predictive(trace, predictions=True, random_seed=42)

forecast_obs = forecast.predictions["obs"]
p_extreme = (forecast_obs > flow_c.std()).mean().item()
print(f"P(Extreme centered flow > 1 sigma) = {p_extreme:.2%}")
foreseeable_threshold = 0.30
is_clause_412_foreseeable = p_extreme > foreseeable_threshold
print(f"Clause 4.12 foreseeable at 30% threshold: {is_clause_412_foreseeable}")
print(f"Foreseeability threshold = {foreseeable_threshold:.0%}")
print(f"Exceedance margin = {(p_extreme - foreseeable_threshold):+.2%}")
clause_412_signal = "Foreseeable" if is_clause_412_foreseeable else "Potentially Unforeseeable"
print(f"Clause 4.12 signal = {clause_412_signal}")
commercial_liability_flag = "HIGH" if is_clause_412_foreseeable else "WATCH"
print(f"Commercial liability flag = {commercial_liability_flag}")
result_message = f"Decision = {clause_412_signal}"
print(result_message)
print("Risk review is complete")

# Persist auditable decision report
report = {"p_extreme": float(p_extreme), "decision": clause_412_signal, "commercial_liability": commercial_liability_flag}
import json
with open('clause4_12_report.json', 'w') as f: f.write(json.dumps(report, indent=2))

with open('clause4_12_report.json', 'r') as f: saved_report = json.load(f)
print(f"Report verified: {saved_report}")
print(f"*** AUDIT TRAIL FINALIZED ***")
print("="*70)
print("AR(1)+Exogenous model execution complete. Posterior inference closed.")
print("="*70)

np.save("forecast_obs_samples.npy", np.asarray(forecast_obs))
report["forecast_samples_file"] = "forecast_obs_samples.npy"
with open('clause4_12_report.json', 'w') as f: f.write(json.dumps(report, indent=2))
report["forecast_mean"] = float(np.asarray(forecast_obs).mean())
report["forecast_std"] = float(np.asarray(forecast_obs).std())
with open('clause4_12_report.json', 'w') as f: f.write(json.dumps(report, indent=2))
report["forecast_p95"] = float(np.percentile(np.asarray(forecast_obs), 95))
with open('clause4_12_report.json', 'w') as f: f.write(json.dumps(report, indent=2))
report["forecast_p05"] = float(np.percentile(np.asarray(forecast_obs), 5))
with open('clause4_12_report.json', 'w') as f: f.write(json.dumps(report, indent=2))